# Playbook 01 — Single-Cell Preprocessing

Load a single-cell AnnData file → QC → clustering → cell-state embedding.
Demonstrates the CPU path (Scanpy). The GPU path (RAPIDS) is identical except for the backend dispatcher.

**Inputs:**
- `.h5ad` file with `obs['condition']` (disease vs control) and `obs['cell_type']`
- `config/single_cell_config.yaml`

**Outputs:**
- `artifacts/single_cell/qc/qc_report.md`
- Filtered + clustered AnnData (in-memory)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
from pathlib import Path
import numpy as np

## 1. Synthetic AnnData (replace with real `.h5ad` for production)

In [ ]:
import anndata as ad
rng = np.random.default_rng(42)
n_cells, n_genes = 800, 2500
X = rng.negative_binomial(5, 0.3, size=(n_cells, n_genes)).astype('float32')
adata = ad.AnnData(X=X)
adata.var_names = [f'GENE_{i:04d}' for i in range(n_genes)]
adata.obs['condition'] = ['disease']*400 + ['control']*400
adata.obs['cell_type'] = (['T_cell']*300 + ['B_cell']*250 + ['NK_cell']*250)
adata.obs['batch'] = ['batch_A']*400 + ['batch_B']*400
print(adata)

## 2. Backend detection + dispatch

In [ ]:
from single_cell_layer.device_detection import detect_backend
from single_cell_layer.loaders import load_single_cell_config

cfg = load_single_cell_config()
backend = detect_backend(prefer_gpu=cfg.get('single_cell', {}).get('prefer_gpu', True))
print(f'Selected backend: {backend}')

## 3. QC: mitochondrial %, doublets, filtering

In [ ]:
from single_cell_layer.mitochondrial_metrics import compute_mito_metrics
from single_cell_layer.filtering import filter_cells_and_genes

compute_mito_metrics(adata)
before = adata.shape
filter_cells_and_genes(adata, min_genes=10, min_cells=3, max_mito_pct=20)
print(f'Filtered {before} → {adata.shape}')

## 4. Dimensionality reduction + clustering (CPU path)

In [ ]:
import scanpy as sc
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=500)
sc.pp.pca(adata, n_comps=30)
sc.pp.neighbors(adata, n_neighbors=15)
sc.tl.leiden(adata, resolution=0.5)
print('Clusters:', adata.obs['leiden'].value_counts().to_dict())

## 5. Write QC report

In [ ]:
from single_cell_layer.qc_report import write_qc_report
out = write_qc_report(adata, out_dir='artifacts/single_cell/qc')
print(f'QC report: {out}')

**Next:** `02_disease_signature_generation.ipynb` consumes the clustered AnnData and emits a disease signature JSON.